### Tools

#### Models can request to call tools that perform taska such as fetching data from a database, searching the web, or running code. Tools are pairing of:

####     1. A schema, including name of the tool, a description, and/or argument definitions (often a JSON schema)
####     2. A function or coroutine to execute


In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("why do parrots talk?")
response

'Parrots don’t “talk” in the same way humans do, but they are exceptionally good at **vocal mimicry**—the ability to copy sounds they hear in their environment. Several biological and social factors make this possible:\n\n| Factor | How it contributes to a parrot’s ability to mimic speech |\n|--------|----------------------------------------------------------|\n| **Highly developed vocal apparatus** | Parrots have a specialized syrinx (the bird equivalent of a larynx) that can produce a wide range of frequencies and rapid modulations, allowing them to reproduce the pitch and rhythm of human words. |\n| **Advanced brain regions for vocal learning** | The “song system” in a parrot’s brain (especially the **nidopallium** and **arcopallium**) is analogous to the human brain areas involved in speech learning. This lets them form auditory memories and reproduce complex sounds. |\n| **Social nature** | In the wild, parrots live in flocks where they constantly exchange calls to maintain group 

In [15]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location.""" # this defition is used by LLM to identify the tool and its purpose
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_weather])

In [19]:
response = model_with_tools.invoke("what is the weather in Noida?")
print(response)
for tool_call in response.tool_calls:
    # view tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")



content='' additional_kwargs={'reasoning_content': 'User asks weather in Noida. Need to call function get_weather.', 'tool_calls': [{'id': 'fc_0a00fc13-0881-4c43-811c-db624ab31b73', 'function': {'arguments': '{"location":"Noida"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 128, 'total_tokens': 171, 'completion_time': 0.089285739, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.00858024, 'prompt_tokens_details': None, 'queue_time': 0.397567701, 'total_time': 0.097865979}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0c808-cb9a-72e1-b1f6-a27b9f1604bf-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Noida'}, 'id': 'fc_0a00fc13-0881-4c43-811c-db624ab31b73', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_toke

### Tool Execution Loops

In [21]:
# Step 1: model generates tool calls

messages = [{"role": "user", "content": "what is the weather in Noida?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: execute tool calls and collect results
for tool_call in ai_msg.tool_calls:
    # execute the tool with generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: model generates final response based on tool results
final_response = model_with_tools.invoke(messages)
print(final_response.text)
print(messages)


The current weather in Noida is sunny, with a temperature of 25 °C. Let me know if you’d like a more detailed forecast (e.g., humidity, wind speed, or a multi‑day outlook)!
[{'role': 'user', 'content': 'what is the weather in Noida?'}, AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is the weather in Noida?" Need to fetch weather via function get_weather. Use location "Noida".', 'tool_calls': [{'id': 'fc_a424a95c-d9ee-435f-8de2-08c2792cfe35', 'function': {'arguments': '{"location":"Noida"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 128, 'total_tokens': 184, 'completion_time': 0.116934808, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.005627315, 'prompt_tokens_details': None, 'queue_time': 0.307845074, 'total_time': 0.122562123}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f73454f048', 'service_tier': 'on_demand', 'finish_reason'